# PTM-Prediction — Fases 1-3b en Colab (GPU)

Ejecuta las fases 1 → 1.5 → 2 → 3 → 3b del pipeline sobre una GPU T4 gratuita, en vez de
CPU local.

**Corre:** saneamiento, extracción de estructura, DeepMVP, DeepPTMPred, fase 3b (vía
secretora, Kinase Library, MeToken, EMNGly, competencia entre PTMs).

**No corre:**
- **Fase 3c** (PyRosetta): CPU-only, no se beneficia de la GPU. Desactivada con
  `FASE_A_ENABLED=false` — ejecútala localmente después, ya sin el cuello de botella de
  las fases anteriores.
- **StackGlyEmbed**: depende del venv de otro proyecto
  (`B-Cell-Epitope-Prediction/StackGlyEmbed`) + pesos de un tercero
  (`scipion-chem-tmbed`). Desactivada (`STACKGLYEMBED_ENABLED=false`); el consenso de
  N-glicosilación funciona igual con DeepMVP + EMNGly. Instrucciones para activarla en la
  Sección 10.

Antes de empezar: **Entorno de ejecución → Cambiar tipo de entorno → GPU**.

El disco de la sesión gratuita ronda 112GB y se llena bastante entre entornos conda y
pesos -- cada celda de entorno limpia su caché al terminar (`mamba clean -afy`), pero si
el panel de "Resources" muestra poco espacio libre, ejecuta esa misma orden manualmente
antes de seguir.


In [ ]:
!nvidia-smi


## 1. Drive (cache de pesos)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CACHE_ROOT = "/content/drive/MyDrive/PTM-Prediction-Colab"
for sub in ["weights/DeepMVP/models",
            "weights/DeepPTMPred/esm",
            "weights/MeToken",
            "weights/EMNgly/esm",
            "weights/EMNgly/checkpoints",
            "outputs_backup"]:
    os.makedirs(f"{CACHE_ROOT}/{sub}", exist_ok=True)
print("Cache en:", CACHE_ROOT)


## 2. `condacolab` (Miniforge + `mamba`)

Esta celda reinicia el runtime automáticamente. Es normal — continúa con la celda
siguiente cuando termine.


In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()


### Continúa aquí tras el reinicio

In [ ]:
import os
# En runtime GPU, Colab fija LD_LIBRARY_PATH=/usr/lib64-nvidia al reiniciar y pisa el
# parche de condacolab -- lo prepende de nuevo antes de check().
os.environ["LD_LIBRARY_PATH"] = "/usr/local/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

import condacolab
condacolab.check()
!mamba --version

CACHE_ROOT = "/content/drive/MyDrive/PTM-Prediction-Colab"
REPO = "/content/PTM-Prediction"
assert os.path.isdir(CACHE_ROOT), "Drive no está montado -- repite la celda de montaje."


## 3. Clonar el pipeline y los 4 repos de motores

In [ ]:
%cd /content
!git clone -q https://github.com/Lvera-code/PTM-Prediction.git

%cd {REPO}
!pip install -q -r requirements.txt

# Clonar los 4 repos antes de crear ninguna subcarpeta de checkpoints: git no clona
# sobre un directorio ya existente y no vacío.
!git clone -q https://github.com/bzhanglab/DeepMVP DeepMVP
!git clone -q https://github.com/kuikui-wang/DeepPTMPred DeepPTMPred
!git clone -q https://github.com/A4Bio/MeToken MeToken
!git clone -q https://github.com/StellaHxy/EMNgly EMNgly
print("5 repos clonados.")


## 4. Descargas pesadas en segundo plano

ESM-1b/ESM-2 se descargan con `aria2c` mientras se crean los entornos conda. Si ya están
en Drive de una sesión anterior, se copian en lugar de descargarse. La Sección 12 espera
a que terminen antes de ejecutar el pipeline.


In [ ]:
!apt-get -qq install -y aria2 > /dev/null
import os
os.makedirs("/content/dl_logs", exist_ok=True)
DL_LOGS = [
    "/content/dl_logs/esm2_main.log",
    "/content/dl_logs/esm2_reg.log",
    "/content/dl_logs/esm1b_main.log",
    "/content/dl_logs/esm1b_reg.log",
    "/content/dl_logs/nglyde_svm.log",
    "/content/dl_logs/metoken_zip.log",
]
for f in DL_LOGS:
    open(f, "w").close()


In [ ]:
%%bash -s "$CACHE_ROOT" "$REPO"
CACHE_ROOT="$1"
REPO="$2"

fetch_or_copy () {
  local tag="$1" drive_path="$2" url="$3" dest="$4" log="$5"
  mkdir -p "$(dirname "$dest")"
  if [ -s "$drive_path" ]; then
    echo "[$tag] copiando desde Drive..." > "$log"
    cp "$drive_path" "$dest"
  else
    echo "[$tag] descargando..." > "$log"
    aria2c -x 16 -s 16 -k 1M -q -o "$(basename "$dest")" -d "$(dirname "$dest")" "$url" >> "$log" 2>&1
    mkdir -p "$(dirname "$drive_path")"
    cp "$dest" "$drive_path"
  fi
  echo "[$tag] listo." >> "$log"
}

fetch_or_copy "esm2-main" \
  "$CACHE_ROOT/weights/DeepPTMPred/esm/esm2_t33_650M_UR50D.pt" \
  "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t33_650M_UR50D.pt" \
  "$REPO/DeepPTMPred/esm/checkpoints/esm2_t33_650M_UR50D.pt" \
  "/content/dl_logs/esm2_main.log" &

fetch_or_copy "esm2-reg" \
  "$CACHE_ROOT/weights/DeepPTMPred/esm/esm2_t33_650M_UR50D-contact-regression.pt" \
  "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t33_650M_UR50D-contact-regression.pt" \
  "$REPO/DeepPTMPred/esm/checkpoints/esm2_t33_650M_UR50D-contact-regression.pt" \
  "/content/dl_logs/esm2_reg.log" &

fetch_or_copy "esm1b-main" \
  "$CACHE_ROOT/weights/EMNgly/esm/esm1b_t33_650M_UR50S.pt" \
  "https://dl.fbaipublicfiles.com/fair-esm/models/esm1b_t33_650M_UR50S.pt" \
  "$REPO/EMNgly/esm/checkpoints/esm1b_t33_650M_UR50S.pt" \
  "/content/dl_logs/esm1b_main.log" &

fetch_or_copy "esm1b-reg" \
  "$CACHE_ROOT/weights/EMNgly/esm/esm1b_t33_650M_UR50S-contact-regression.pt" \
  "https://dl.fbaipublicfiles.com/fair-esm/regression/esm1b_t33_650M_UR50S-contact-regression.pt" \
  "$REPO/EMNgly/esm/checkpoints/esm1b_t33_650M_UR50S-contact-regression.pt" \
  "/content/dl_logs/esm1b_reg.log" &

fetch_or_copy "nglyde-svm" \
  "$CACHE_ROOT/weights/EMNgly/checkpoints/N-GlyDE.pickle" \
  "https://drive.usercontent.google.com/download?id=1hbnEtHHXTGnQAFm-cCHMj3pWQiAYAUsw&export=download&confirm=t" \
  "$REPO/EMNgly/checkpoints/N-GlyDE.pickle" \
  "/content/dl_logs/nglyde_svm.log" &

(
  if [ -s "$CACHE_ROOT/weights/MeToken/pretrained_model.zip" ]; then
    echo "[metoken-zip] copiando desde Drive..." > /content/dl_logs/metoken_zip.log
    cp "$CACHE_ROOT/weights/MeToken/pretrained_model.zip" /tmp/pretrained_model.zip
  else
    echo "[metoken-zip] descargando..." > /content/dl_logs/metoken_zip.log
    curl -sL -o /tmp/pretrained_model.zip \
      https://github.com/A4Bio/MeToken/releases/download/1.0/pretrained_model.zip
    cp /tmp/pretrained_model.zip "$CACHE_ROOT/weights/MeToken/pretrained_model.zip"
  fi
  echo "[metoken-zip] listo." >> /content/dl_logs/metoken_zip.log
) &

disown -a
echo "6 descargas arrancadas en segundo plano."
echo "(Progreso: !tail -n2 /content/dl_logs/*.log)"


## 5. DeepMVP

Los pesos (~1.6GB) no tienen URL directa (web con clic manual en `deepmvp.ptmax.org`).
Súbelos una vez a `Drive/PTM-Prediction-Colab/weights/DeepMVP/models/` -- las siguientes
sesiones ya los encuentran ahí.


In [ ]:
%cd {REPO}
# pyteomics=4.4.2 no existe en conda-forge (solo en PyPI) -- se instala por separado en
# vez de usar environment.yml directo, que falla al resolver por ese paquete.
!mamba create -q -n deepmvp python=3.7.10 -y
!mamba install -q -n deepmvp -c conda-forge -y ipython numpy=1.19.5 h5py=2.10.0 pandas=1.2.4 scikit-learn=0.24.2 matplotlib=3.4.2 biopython=1.78 shap=0.39.0 cudatoolkit=11.0 "cudnn=8.0.*"
!mamba run -n deepmvp pip install -q pyteomics==4.4.2 tensorflow==2.4.2
!mamba clean -afy -q

DEEPMVP_PYTHON_BIN = !mamba run -n deepmvp which python
DEEPMVP_PYTHON_BIN = DEEPMVP_PYTHON_BIN[0]
print("DEEPMVP_PYTHON_BIN =", DEEPMVP_PYTHON_BIN)


In [ ]:
import os, shutil
drive_models = f"{CACHE_ROOT}/weights/DeepMVP/models"
local_models = f"{REPO}/DeepMVP/models"

if os.listdir(drive_models):
    shutil.copytree(drive_models, local_models, dirs_exist_ok=True)
    print("Pesos copiados:", os.listdir(local_models))
else:
    print(f"No hay pesos en Drive todavía. Súbelos a: {drive_models}")
    print("DeepMVP queda sin pesos por ahora (degrada solo, no rompe el resto).")


## 6. DeepPTMPred

In [ ]:
%cd {REPO}
!mamba env create -q -f DeepPTMPred/pred/train_PTM/environment.yml -n deepptmpred
!mamba clean -afy -q

DEEPPTMPRED_PYTHON_BIN = !mamba run -n deepptmpred which python
DEEPPTMPRED_PYTHON_BIN = DEEPPTMPRED_PYTHON_BIN[0]
print("DEEPPTMPRED_PYTHON_BIN =", DEEPPTMPRED_PYTHON_BIN)


## 7. MeToken

In [ ]:
%cd {REPO}
!mamba create -q -n metoken python=3.10 -y
!mamba run -n metoken pip install -q torch==2.4.0 --index-url https://download.pytorch.org/whl/cu121
!mamba run -n metoken pip install -q numpy scipy biopython transformers omegaconf tqdm pandas huggingface-hub h5py

!mamba run -n metoken pip install -q torch_scatter -f https://data.pyg.org/whl/torch-2.4.0+cu121.html \
  || (echo "Sin wheel prebuilt -- compilando desde fuente." && mamba run -n metoken pip install -q torch_scatter)
!mamba clean -afy -q
!mamba run -n metoken pip cache purge -q

METOKEN_PYTHON_BIN = !mamba run -n metoken which python
METOKEN_PYTHON_BIN = METOKEN_PYTHON_BIN[0]
print("METOKEN_PYTHON_BIN =", METOKEN_PYTHON_BIN)


## 8. EMNGly

Los pesos del SVM se entrenaron con `scikit-learn==1.1.1` -- no actualizar esa versión.


In [ ]:
%cd {REPO}
!python3 -m venv .venv-emngly
!.venv-emngly/bin/pip install -q torch==2.4.0 --index-url https://download.pytorch.org/whl/cu121
!.venv-emngly/bin/pip install -q fair-esm "scikit-learn==1.1.1" scipy pandas tqdm wget
!.venv-emngly/bin/pip install -q "numpy==1.23.5"
!.venv-emngly/bin/pip cache purge -q
EMNGLY_PYTHON_BIN = f"{REPO}/.venv-emngly/bin/python"
print("EMNGLY_PYTHON_BIN =", EMNGLY_PYTHON_BIN)


## 9. Kinase Library

In [ ]:
!mamba create -q -n kinase_library python=3.10 -y
!mamba run -n kinase_library pip install -q kinase-library
!mamba clean -afy -q

KINASE_LIBRARY_PYTHON_BIN = !mamba run -n kinase_library which python
KINASE_LIBRARY_PYTHON_BIN = KINASE_LIBRARY_PYTHON_BIN[0]
print("KINASE_LIBRARY_PYTHON_BIN =", KINASE_LIBRARY_PYTHON_BIN)


## 10. StackGlyEmbed (opcional, desactivado)

Para activarla, clona `Lvera-code/BCell-Epitope-Prediction` y `scipion-chem-tmbed`, monta
su venv (Sección 11 del README de ese proyecto) y define esto antes de la Sección 11:

```python
os.environ["STACKGLYEMBED_ENABLED"] = "true"
os.environ["STACKGLYEMBED_PYTHON_BIN"] = f"{REPO}/BCell-Epitope-Prediction/StackGlyEmbed/.venv-stackglyembed/bin/python"
os.environ["STACKGLYEMBED_MODELS_DIR"] = f"{REPO}/BCell-Epitope-Prediction/StackGlyEmbed/prediction"
os.environ["STACKGLYEMBED_T5_MODEL_PATH"] = f"{REPO}/scipion-chem-tmbed/tmbed_src/tmbed/models/t5"
```


## 11. Variables de entorno

In [ ]:
import os

os.environ["DEEPMVP_PYTHON_BIN"] = DEEPMVP_PYTHON_BIN
os.environ["DEEPMVP_HOME"] = f"{REPO}/DeepMVP"
os.environ["DEEPMVP_MODEL_DIR"] = f"{REPO}/DeepMVP/models"

os.environ["DEEPPTMPRED_PYTHON_BIN"] = DEEPPTMPRED_PYTHON_BIN
os.environ["DEEPPTMPRED_HOME"] = f"{REPO}/DeepPTMPred"

os.environ["METOKEN_PYTHON_BIN"] = METOKEN_PYTHON_BIN
os.environ["METOKEN_HOME"] = f"{REPO}/MeToken"
os.environ["METOKEN_ENABLED"] = "true"

os.environ["EMNGLY_PYTHON_BIN"] = EMNGLY_PYTHON_BIN
os.environ["EMNGLY_HOME"] = f"{REPO}/EMNgly"
os.environ["EMNGLY_ENABLED"] = "true"

os.environ["KINASE_LIBRARY_PYTHON_BIN"] = KINASE_LIBRARY_PYTHON_BIN
os.environ["KINASE_LIBRARY_ENABLED"] = "true"

os.environ["STACKGLYEMBED_ENABLED"] = "false"
os.environ["FASE_A_ENABLED"] = "false"

os.environ["FASTA_INPUT_DIR"] = f"{REPO}/inputs"
os.environ["FASTA_OUTPUT_DIR"] = f"{REPO}/outputs"

print("Variables de entorno listas.")


## 12. Esperar las descargas en segundo plano

In [ ]:
import time

def wait_for_downloads(logs, poll=15, timeout=3600):
    start = time.time()
    pending = set(logs)
    while pending:
        done_now = {log for log in pending if os.path.exists(log) and "listo." in open(log).read()}
        pending -= done_now
        if not pending:
            break
        if time.time() - start > timeout:
            raise TimeoutError(f"Timeout esperando: {pending}")
        print(f"[{int(time.time() - start)}s] esperando {len(pending)}/{len(logs)} descarga(s)...")
        time.sleep(poll)
    print("Descargas completas.")

wait_for_downloads(DL_LOGS)


## 13. Extraer los pesos de MeToken

In [ ]:
import zipfile
zipfile.ZipFile('/tmp/pretrained_model.zip').extractall(REPO + '/MeToken')
!ls {REPO}/MeToken/pretrained_model/


## 14. Elegir el input a ejecutar

Un input = un archivo FASTA o PDB/mmCIF, con una o varias secuencias dentro (todas se
procesan en la misma ejecución). Elige un candidato del panel o sube el tuyo -- cada
ejecución procesa solo el que esté seleccionado aquí.


In [ ]:
import os
from google.colab import files

candidato = "p53_P04637.pdb"  #@param ["p53_P04637.pdb", "hif1a_Q16665.pdb", "histone_h3_P68431.pdb", "histone_h4_P62805.pdb", "prothrombin_P00734.pdb", "epo_P01588.pdb", "kit_ligand_scf_P21583.pdb", "(subir un archivo nuevo)"]

if candidato == "(subir un archivo nuevo)":
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    os.rename(fname, f"{REPO}/inputs/{fname}")
    input_file = f"{REPO}/inputs/{fname}"
else:
    input_file = f"{REPO}/inputs/{candidato}"

print("Input seleccionado:", input_file)


## 15. Ejecutar

In [ ]:
%cd {REPO}
!python pipeline.py --input {input_file} --output-dir outputs


## 16. Resultados

In [ ]:
import glob, pandas as pd

report = sorted(glob.glob(f"{REPO}/outputs/*_ptm_sites.csv"), key=os.path.getmtime)[-1]
df = pd.read_csv(report)
print(f"{len(df)} sitio(s) en {report}")
df.sort_values("posicion").head(30)


In [ ]:
import shutil
shutil.copy(report, f"{CACHE_ROOT}/outputs_backup/{os.path.basename(report)}")
print("Copia en Drive:", f"{CACHE_ROOT}/outputs_backup/{os.path.basename(report)}")


## Fase 3c (local)

Descarga el CSV (o la copia de `outputs_backup/` en Drive) y ejecútalo localmente con tu
entorno `deepptmpred` (PyRosetta) ya instalado -- ahora es rápido, el cuello de botella
de fases 1-3b ya está resuelto.
